# Simplified Pipeline — PM2.5 in Kampala and Nairobi

**By the end of this notebook, you will be able to:**
- Train a scikit-learn model using the `fit`/`predict` pattern shared by (almost) every model in the library
- Evaluate a model by testing it on a city it never trained on, instead of a random split
- Explain, concretely, why a raw geographic coordinate can break a linear model

Goal: build, end to end, the smallest complete and methodologically correct Machine Learning pipeline: load, clean, split, engineer features, train a model, evaluate. Each step is a block of code to complete. The train/test split principle here is deliberately simple: **train on one city, test on the other**.

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TRAIN_CITY = "Kampala"
TEST_CITY = "Nairobi"
COLUMNS = [
    "city", "date", "hour", "site_latitude", "site_longitude", "pm2_5",
    "sulphurdioxide_so2_column_number_density",
    "carbonmonoxide_co_column_number_density",
    "nitrogendioxide_no2_column_number_density",
    "formaldehyde_tropospheric_hcho_column_number_density",
    "uvaerosolindex_absorbing_aerosol_index",
    "ozone_o3_column_number_density",
    "uvaerosollayerheight_aerosol_height",
    "cloud_cloud_fraction",
]

## Step 1 — Load and restrict the scope

Load `data/train.csv`, keep only the two cities and the columns listed above. Then, so the pipeline stays fast to run and easy to inspect, cap the sample at 1200 rows per city.

**Documentation references:**
- [`Series.isin()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html)
- [`DataFrame.sample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html)

**Exercise:** Filter the raw data to the two cities and the listed columns. Then, city by city, draw a random sample of at most 1200 rows — pass a fixed `random_state` so the sample, and every metric computed from it later, stays reproducible between runs. Combine the two samples back into one table.

In [15]:
# TODO 1: load train.csv, filter the two cities and the columns, cap at 1200 rows/city
train_data_raw = pd.read_csv("../../data/train.csv")
train_data_right_columns = train_data_raw[COLUMNS]

cities = [TRAIN_CITY, TEST_CITY]

train_data_right_columns_and_cities = train_data_right_columns[train_data_right_columns["city"].isin(cities)]
train_data_right_columns_and_cities

samples = []
for city in cities:
    train_data_city = train_data_right_columns_and_cities[train_data_right_columns_and_cities["city"]==city]
    samples.append(train_data_city.sample(n = min(len(train_data_city), 1200), random_state = 67))

train_data = pd.concat(samples, ignore_index=True)
train_data

,city,date,hour,site_latitude,site_longitude,pm2_5,sulphurdioxide_so2_column_number_density,carbonmonoxide_co_column_number_density,nitrogendioxide_no2_column_number_density,formaldehyde_tropospheric_hcho_column_number_density,uvaerosolindex_absorbing_aerosol_index,ozone_o3_column_number_density,uvaerosollayerheight_aerosol_height,cloud_cloud_fraction
0,Kampala,2023-05-24,11,0.392050,32.672960,7.445700,0.000240,0.028297,0.000078,0.000326,-0.426318,0.120995,NaN,NaN
1,Kampala,2023-12-01,11,0.347014,32.649360,33.261000,NaN,0.041357,NaN,NaN,0.175395,0.111937,NaN,0.549944
2,Kampala,2023-08-21,11,0.333529,32.568643,24.518900,NaN,NaN,NaN,NaN,0.849702,0.122248,NaN,0.593274
3,Kampala,2023-06-20,11,0.322320,32.575700,22.756800,NaN,0.033802,NaN,NaN,-1.060516,0.115828,NaN,0.796085
4,Kampala,2023-06-09,11,0.344000,32.553000,21.312000,NaN,0.027246,NaN,0.000125,-0.279367,0.121178,NaN,0.354141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2395,Nairobi,2023-12-31,10,-1.338275,36.741783,12.224615,-0.000006,0.027858,NaN,0.000186,-1.161002,0.108206,NaN,0.222727
2396,Nairobi,2023-05-18,11,-1.224292,36.805005,21.679012,NaN,0.023818,NaN,NaN,0.238854,0.121240,NaN,0.641067
2397,Nairobi,2023-06-25,11,-1.283724,36.810364,23.917000,0.000285,0.028733,0.000112,0.000318,-0.994782,0.118627,NaN,0.143885
2398,Nairobi,2024-01-24,11,-1.338275,36.741783,17.163500,-0.000152,0.034562,0.000075,0.000113,-0.255104,0.106024,NaN,NaN


## Step 2 — Clean missing values

Some satellite measurements are missing for some passes — you already measured how much, per city, in the discovery notebook. Fill them **within each city**, sorted by date: never let one city's values fill another city's gaps, that would be a geographic information leak.

**Documentation references:**
- [`DataFrameGroupBy.transform()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.GroupBy.transform.html)
- [`Series.ffill()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.ffill.html) / [`Series.bfill()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.bfill.html)

**Exercise:** Sort the rows by city and date. Then, for each fillable column, group by city and use `transform` to apply forward-fill followed by backward-fill *within* each group — `transform` is what lets you get a filled column back in the original row order, ready to reassign to `df`, rather than one small table per city to reassemble yourself.

In [34]:
# TODO 2: fill missing values by city, sorted by date
train_data["date"] = pd.to_datetime(train_data["date"])
missing_value_by_city_by_date = (train_data.groupby(["city", "date"]).apply(lambda group: group.isna().mean()).sort_index())
train_data = train_data.sort_values(["city", "date"])
print(train_data)

         city       date  hour  site_latitude  site_longitude      pm2_5  \
70    Kampala 2023-01-02    10       0.381576       32.647109  18.386100   
281   Kampala 2023-01-02    10       0.308520       32.592240  12.529800   
338   Kampala 2023-01-02    10       0.292820       32.571320  20.839100   
752   Kampala 2023-01-02    10       0.347014       32.649360  17.765300   
833   Kampala 2023-01-02    10       0.374653       32.557440  17.506000   
...       ...        ...   ...            ...             ...        ...   
1573  Nairobi 2024-01-27    10      -1.283724       36.810364  13.334286   
1665  Nairobi 2024-01-27    10      -1.268877       36.819139  13.103000   
1877  Nairobi 2024-01-27    10      -1.243396       36.886487  12.566731   
2030  Nairobi 2024-01-27    10      -1.311080       36.861865  11.920556   
2376  Nairobi 2024-01-27    10      -1.287094       36.821671  17.412500   

      sulphurdioxide_so2_column_number_density  \
70                                   

/var/folders/7d/mtnr2ts91fj4fz94jfr927880000gn/T/ipykernel_94738/901378847.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  missing_value_by_city_by_date = (train_data.groupby(["city", "date"]).apply(lambda group: group.isna().mean()).sort_index())


## Step 3 — Manual split by city

A basic way to check whether a model actually generalizes is to test it on data it has never seen. Here, the simplest version of that idea: train on `TRAIN_CITY`, evaluate on `TEST_CITY` — no random shuffling of rows. That is enough to get started in this activity. Later on, you will see a more rigorous and systematic way to do this across several groups at once, and to guard against overfitting — cross-validation, covered in a dedicated Session 3 module.

**Exercise:** Split `df` into `train_df` (rows where `city == TRAIN_CITY`) and `test_df` (rows where `city == TEST_CITY`).

In [4]:
# TODO 3: split into train_df (TRAIN_CITY) and test_df (TEST_CITY)


## Step 4 — Temporal features

A model can only exploit the patterns it can actually see in its input. A raw date doesn't expose seasonality or a weekly rhythm — nothing in a linear model lets it notice "this is a rainy month" or "this is a weekday" just by looking at a timestamp. When you know, from domain knowledge, that a pattern is likely to matter — here, pollution following seasonal and weekly cycles from traffic and economic activity — it is worth engineering it explicitly as a feature rather than hoping the model finds it on its own. That is why the next step extracts the month and the day of week from `date`.

**Exercise:** For both `train_df` and `test_df`, add `month` and `dayofweek` columns from `date`, using the same `.dt` accessor you already used in the discovery notebook.

In [5]:
# TODO 4: add month and dayofweek columns derived from date


## Step 5 — Baseline model and evaluation

Every scikit-learn model follows the same two-step contract, no matter how simple or advanced: `model.fit(X_train, y_train)` learns from data, `model.predict(X_test)` applies what it learned to new data. You will reuse this exact pattern for every model in this course, starting with the simplest one, `LinearRegression`.

**Warning**: `site_latitude`/`site_longitude` are deliberately excluded from the features. In the training sample, one city only occupies a small geographic area (a handful of very close monitoring sites): a linear model fits a huge coefficient on these coordinates, which explodes as soon as it is applied to very different coordinates (the other city). Try adding them back to `feature_cols` to see the effect on the metrics.

For a refresher on what RMSE, MAE, and R² actually measure, see [section 6.2 of Machine Learning Techniques: From Theory to Practice](https://hub.imt-atlantique.fr/datascience-toolkit/courses/lesson_1/#62-evaluation-metrics-for-regression).

**Documentation references:**
- [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [`mean_squared_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html), [`mean_absolute_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html), [`r2_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)

**Exercise:** Build `feature_cols`: every numeric column of `train_df` except `pm2_5` (the target) and the columns named in the warning above. Fit a `LinearRegression` on `train_df`, predict on `test_df`, then compute RMSE (the square root of `mean_squared_error`), MAE, and R².

In [6]:
# TODO 5: define the features, train the linear model, compute RMSE/MAE/R2


## Going further

What would happen if you reversed it: train on Nairobi and test on Kampala? Try it in the cell below and compare the two errors.

In Session 3, you will see how to automate this kind of comparison across more than two cities at once with `GroupKFold` — see [section 4.2.3](https://hub.imt-atlantique.fr/datascience-toolkit/courses/lesson_1/#423-groupkfold-cross-validation) for a preview.

**Continue on the site:** [Build a First Pipeline](https://hub.imt-atlantique.fr/datascience-toolkit/session2/practical_2/)